# NB-1: LoCoMo — CSAM vs Flat-RAG Baseline (All 4 Models)

Runs CSAM (3-tier hierarchical memory with consolidation-aware forgetting) and a Flat-RAG baseline
(L2-only, no knowledge graph) on the LoCoMo long-conversation QA dataset.

**Models:** Llama-3.1-8B · Llama-4-Scout-17B · Llama-3.3-70B · GPT-OSS-120B  
**Output directory:** `results/nb1_locomo/`  
**Time estimate:** ~20–40 min per model pair (5 conversations each)

**Just run all cells top-to-bottom. No interaction needed after cell 3.**

## Step 1 — Install dependencies & clone repo

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)
print('Deps installed')

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    r = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                       capture_output=True, text=True)
    print(r.stdout.strip() or 'Already up to date')
    if r.returncode != 0:
        print('[WARN] pull failed:', r.stderr[:200])
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

commit = subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, cwd=REPO_DIR)
print(f'Commit: {commit.stdout.strip()}')
print(f'Working dir: {os.getcwd()}')

## Step 2 — Set API keys

**Kaggle:** Notebook → Settings → Secrets → add `GROQ_API_KEY` (and optionally `GROQ_API_KEY_2` … `GROQ_API_KEY_5`)  
**Colab:** Left sidebar key icon → add same secrets

In [ ]:
import os

def _load_secret(name: str) -> str:
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v: return v
    except Exception: pass
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v: return v
    except Exception: pass
    return os.environ.get(name, '')

env_lines = []
key = _load_secret('GROQ_API_KEY')
if not key:
    raise RuntimeError('GROQ_API_KEY not found — add it to Kaggle/Colab Secrets')
os.environ['GROQ_API_KEY'] = key
env_lines.append(f'GROQ_API_KEY={key}')
print('GROQ_API_KEY loaded')

for i in range(2, 10):
    k = _load_secret(f'GROQ_API_KEY_{i}')
    if k:
        os.environ[f'GROQ_API_KEY_{i}'] = k
        env_lines.append(f'GROQ_API_KEY_{i}={k}')
        print(f'GROQ_API_KEY_{i} loaded')

with open('.env', 'w') as f:
    f.write('\n'.join(env_lines) + '\n')
print(f'\n.env written with {len(env_lines)} key(s)')
print('Key rotation active — 429 errors will auto-rotate to next key')

## Step 3 — Configure
Edit these values before running. Defaults are publication-quality settings.

In [ ]:
import os

# ── EDIT THESE IF NEEDED ────────────────────────────────────────────────────
MAX_CONVERSATIONS = 5      # 3 for quick test, 5-10 for publication
QUESTIONS_PER_CONV = None  # None = all questions per conversation
SEED = 42
CHECKPOINT_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
# ─────────────────────────────────────────────────────────────────────────────

DATASET = os.path.join(REPO_DIR, 'csam_project', 'benchmarks', 'data', 'locomo10.json')
OUT_DIR = os.path.join(REPO_DIR, 'results', 'nb1_locomo')
os.makedirs(OUT_DIR, exist_ok=True)

print(f'Dataset:      {DATASET}')
print(f'Dataset OK:   {os.path.exists(DATASET)}')
print(f'Output dir:   {OUT_DIR}')
print(f'Conversations:{MAX_CONVERSATIONS}')
print(f'Seed:         {SEED}')

if not os.path.exists(DATASET):
    data_dir = os.path.dirname(DATASET)
    print(f'[WARN] Dataset missing. Files in {data_dir}:')
    try: print(os.listdir(data_dir))
    except Exception as e: print(f'  {e}')

## Step 4 — Run CSAM benchmark (all 4 models)
Runs the 3-tier CSAM system (L1 LRU + L2 HNSW + L3 knowledge graph with consolidation-aware forgetting).

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_multimodel',
    '--all',
    '--consolidate',
    '--dataset', DATASET,
    '--max-conversations', str(MAX_CONVERSATIONS),
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
if QUESTIONS_PER_CONV:
    cmd += ['--questions-per-conv', str(QUESTIONS_PER_CONV)]

print('Running CSAM (all 4 models)...')
print(f'CMD: {" ".join(cmd[2:])}\n')
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print('[FAIL] CSAM benchmark failed — check output above')
else:
    print(f'\n[OK] CSAM benchmark complete. Results in {OUT_DIR}/')

## Step 5 — Run Flat-RAG Baseline (all 4 models)

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_baseline_rag_hosted',
    '--all',
    '--dataset', DATASET,
    '--max-conversations', str(MAX_CONVERSATIONS),
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT_DIR,
    '--output-dir', OUT_DIR,
]
if QUESTIONS_PER_CONV:
    cmd += ['--questions-per-conv', str(QUESTIONS_PER_CONV)]

print('Running Flat-RAG Baseline (all 4 models)...')
result = subprocess.run(cmd, capture_output=False, text=True)
if result.returncode != 0:
    print('[FAIL] Baseline failed — check output above')
else:
    print(f'\n[OK] Baseline complete. Results in {OUT_DIR}/')

## Step 6 — Results summary

In [ ]:
import json, os, glob

csam_files = sorted(glob.glob(os.path.join(OUT_DIR, 'results_locomo_csam_*.json')))
base_files = sorted(glob.glob(os.path.join(OUT_DIR, 'results_locomo_baseline_*.json')))

print('=' * 70)
print('CSAM vs BASELINE — LoCoMo Results')
print('=' * 70)
print(f'{"Model":<28} {"System":<10} {"Macro F1":>10} {"Micro F1":>10} {"Sem Sim":>10}')
print('-' * 70)

def show_row(fp, tag):
    if not os.path.exists(fp): return
    with open(fp) as f: d = json.load(f)
    name = d.get('display_name', d.get('model', '?'))[:27]
    mf1  = d.get('macro_f1', 0)
    uf1  = d.get('micro_f1', 0)
    sem  = d.get('avg_semantic_sim', 0)
    print(f'{name:<28} {tag:<10} {mf1:>10.4f} {uf1:>10.4f} {sem:>10.4f}')

for fp in csam_files: show_row(fp, 'CSAM')
for fp in base_files: show_row(fp, 'Baseline')

print(f'\nFiles in {OUT_DIR}:')
for fp in sorted(os.listdir(OUT_DIR)): print(f'  {fp}')

## Step 7 — Save / Download results

In [ ]:
import shutil, os, glob

all_files = glob.glob(os.path.join(OUT_DIR, '*.json'))

if os.path.exists('/kaggle'):
    kaggle_out = '/kaggle/working/nb1_locomo'
    os.makedirs(kaggle_out, exist_ok=True)
    for fp in all_files:
        dest = os.path.join(kaggle_out, os.path.basename(fp))
        shutil.copy(fp, dest)
        print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in all_files:
            files.download(fp)
            print(f'Downloaded: {fp}')
    except ImportError:
        print('Files saved at:')
        for fp in sorted(all_files): print(f'  {fp}')